# Supa Strikers — player detection & tracking (Kaggle, T4 ×2)

Detects, tracks and team-labels every player, goalkeeper and referee in the **tactical** video for the
two halves only (frames 11,185–90,651 and 110,808–190,197 = 158,857 frames), then scores 9 tracking
variants (3 trackers × 3 team classifiers, each with post-processing) against the ATD baseline.

**Inputs** (Add Input → your datasets): `bar-ath-tactical` (video, ATD txt+xml, homography csv),
`supa-strikers-models` (the original YOLOv8 player model `.pt`), optionally `supa-strikers-outputs`
(a previous run's `/kaggle/working`, only to resume).
**Settings:** Accelerator *GPU T4 ×2*, Internet *on*.

**How to run:** leave `RUN_MODE = "smoke"` and *Run All* (~10–15 min) → check the calibration crop grids,
the alignment line and the scorecard → set `RUN_MODE = "full"` → *Save Version → Save & Run All*.
Nothing in this notebook uses hand labels: teams are named from the ATD's team membership and
goalkeepers by side at kickoff (RULES.md).

In [ ]:
# ===================== CONFIG — everything you might change lives here =====================
import os

RUN_MODE = "smoke"          # "smoke": first SMOKE_FRAMES of each half (~10-15 min) to prove the
                            # whole pipeline end to end. Then set "full" and run all again.
SMOKE_FRAMES = 2000

# Detection
IMGSZ = 1280                # inference size; use what the player model was trained at
BATCH = 16                  # frames per forward pass
DET_CONF = 0.10             # keep low-confidence boxes: ByteTrack's 2nd association uses them
CHUNK = 5000                # frames per Parquet chunk (resume granularity)
WARMUP = 25                 # extra frames before each half so IDs are stable at kickoff

# Team classification
TEAM_EVERY = 3              # score team on every Nth frame (per-track vote needs samples, not all)
USE_SIGLIP = True           # T2 variant; needs Internet on to download weights
CALIB_FRAMES_PER_HALF = 75  # frames sampled to fit the frozen team models

# Tracking / post-processing
MIN_TRACK_DETS = 5          # tracks with fewer detections are dropped (all variants)
STITCH_MAX_GAP = 75         # frames (3 s)
STITCH_MAX_SPEED = 8.0      # m/s allowed while unseen
SMOOTH_WINDOW = 9           # Savitzky-Golay window (frames)
JUMP_SPEED = 12.0           # m/s; faster steps count as jumps

# Outputs
SAVE_ALL_VARIANTS = False   # True: also write every variant's tracks (~100 MB each)
BEST_VARIANT = None         # e.g. "K2-T1+post" to override the automatic pick
MAKE_CLIPS = True
N_CLIPS, CLIP_SECONDS = 6, 20

# Match facts (DATA.md)
FPS = 25
PITCH = (105.3, 68.0)       # metres
N_VIDEO = 199319
HALVES = [(1, 11185, 90651), (2, 110808, 190197)]   # (half, first, last) tactical video frames

# Paths (env vars let the same notebook run outside Kaggle)
DATA_DIR = os.environ.get("SS_DATA_DIR", "/kaggle/input")
WORK = os.environ.get("SS_WORK", "/kaggle/working")
MODEL_PATH = os.environ.get("SS_MODEL")   # None -> the .pt inside the supa-strikers-models dataset

os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)                      # helper modules below are written here and imported from here
import sys
if WORK not in sys.path:
    sys.path.insert(0, WORK)


In [ ]:
import subprocess, sys

if not os.environ.get("SS_SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ultralytics", "supervision==0.25.1", "umap-learn", "pyarrow"], check=False)

import glob, json, math, shutil, time, gc
import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.signal import savgol_filter

print("torch", torch.__version__, "| CUDA devices:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if torch.cuda.device_count() < 2:
    print("NOTE: fewer than 2 GPUs visible — set Accelerator to 'GPU T4 x2' for the full run.")


## Helper modules
Written to `/kaggle/working` so the GPU worker processes (one per T4) can import them.

In [ ]:
%%writefile common.py
"""Shared code for the notebook and the GPU detection workers.

Roles, torso crops, colour histograms, batched ResNet18 / SigLIP embeddings and the frozen team
models fitted at calibration. Imported by det_worker.py (one process per GPU) and by the notebook.
"""
import json
import os

import cv2
import numpy as np
import torch
import torch.nn.functional as F

ROLE_PLAYER, ROLE_GK, ROLE_REF = 0, 1, 2
ROLE_NAME = {ROLE_PLAYER: "player", ROLE_GK: "goalkeeper", ROLE_REF: "referee"}
HOME, AWAY, NO_TEAM = 0, 1, -1
TEAM_LETTER = {HOME: "H", AWAY: "A", NO_TEAM: ""}

# Torso band (fraction of box height) and box sanity limits, from DeepTeamClassifier.
TORSO_Y = (0.15, 0.50)
MIN_W, MIN_H = 12, 25
AR_MIN, AR_MAX = 1.0, 5.0       # height / width
COLOR_WEIGHT = 2.0              # weight of the HSV histogram inside the T1 feature


def role_map_from_names(names):
    """{model class id: role}. Ball / pitch classes are ignored.

    A plain COCO model only has 'person': it is treated as player (referees then end up in the
    player pool and show up in the metrics as overflow).
    """
    out = {}
    for k, v in dict(names).items():
        n = str(v).lower().strip()
        if "ball" in n or "pitch" in n:
            continue
        if "goal" in n or "keeper" in n or n == "gk":
            out[int(k)] = ROLE_GK
        elif "ref" in n:
            out[int(k)] = ROLE_REF
        elif "player" in n or n == "person":
            out[int(k)] = ROLE_PLAYER
    return out


def torso_crop(frame, box):
    """BGR torso crop of one box, or None when the box can't be trusted."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(round(float(v))) for v in box]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
    bw, bh = x2 - x1, y2 - y1
    if bw < MIN_W or bh < MIN_H:
        return None
    ar = bh / bw
    if ar < AR_MIN or ar > AR_MAX:
        return None
    ty1, ty2 = y1 + int(bh * TORSO_Y[0]), y1 + int(bh * TORSO_Y[1])
    if ty2 <= ty1:
        return None
    crop = frame[ty1:ty2, x1:x2]
    return crop if crop.size else None


def hsv_hist(crop_bgr):
    """16x8 Hue/Saturation histogram, L2-normalised (128 values). Value is left out on purpose."""
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [16, 8], [0, 180, 0, 256]).flatten().astype(np.float32)
    n = np.linalg.norm(hist)
    return hist / n if n > 1e-8 else hist


def letterbox(crop_bgr, size=224):
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    sq = cv2.copyMakeBorder(rgb, top, s - h - top, left, s - w - left, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return cv2.resize(sq, (size, size), interpolation=cv2.INTER_LINEAR)


def l2n(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, 1e-8)


def t1_features(hist, rn):
    return l2n(np.concatenate([COLOR_WEIGHT * hist, rn], axis=1)).astype(np.float32)


class Embedder:
    """Batched ResNet18 (ImageNet) and, optionally, SigLIP embeddings of torso crops."""

    def __init__(self, device=None, use_siglip=True, siglip_name="google/siglip-base-patch16-224"):
        from torchvision.models import resnet18, ResNet18_Weights

        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        self.half = self.device.startswith("cuda")
        m = resnet18(weights=None if os.environ.get("SS_NO_WEIGHTS") else ResNet18_Weights.DEFAULT)
        m.fc = torch.nn.Identity()
        self.resnet = m.to(self.device).eval()
        if self.half:
            self.resnet.half()
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=self.device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=self.device).view(1, 3, 1, 1)
        self.siglip = None
        if use_siglip:
            try:
                from transformers import SiglipVisionModel
                self.siglip = SiglipVisionModel.from_pretrained(siglip_name).to(self.device).eval()
                if self.half:
                    self.siglip.half()
            except Exception as e:  # no internet, old transformers ... -> T2 is skipped
                print("SigLIP unavailable, T2 disabled:", repr(e)[:200])

    @torch.no_grad()
    def embed(self, crops, bs=256):
        """crops: list of BGR arrays -> (resnet (n,512), siglip (n,768) or None)."""
        imgs = [letterbox(c) for c in crops]
        rn_out, sg_out = [], []
        for i in range(0, len(imgs), bs):
            x = torch.from_numpy(np.stack(imgs[i:i + bs])).to(self.device).permute(0, 3, 1, 2).float() / 255.0
            xr = (x - self.mean) / self.std
            if self.half:
                xr = xr.half()
            rn_out.append(F.normalize(self.resnet(xr).float(), dim=1).cpu().numpy())
            if self.siglip is not None:
                xs = (x - 0.5) / 0.5
                if self.half:
                    xs = xs.half()
                s = self.siglip(pixel_values=xs).pooler_output.float()
                sg_out.append(F.normalize(s, dim=1).cpu().numpy())
        rn = np.concatenate(rn_out) if rn_out else np.zeros((0, 512), np.float32)
        sg = np.concatenate(sg_out) if sg_out else None
        return rn, sg


def kmeans_p0(X, centers):
    """Soft membership of cluster 0 from distances to two centres."""
    d = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    return d[:, 1] / (d[:, 0] + d[:, 1] + 1e-9)


class TeamModels:
    """Frozen team classifiers. predict() returns P(home) per crop for t1, t2, t3."""

    def __init__(self, arrays):
        self.a = {k: np.asarray(v) for k, v in arrays.items()}

    @classmethod
    def load(cls, path):
        with np.load(path) as z:
            return cls({k: z[k] for k in z.files})

    def save(self, path):
        np.savez(path, **self.a)

    def _home(self, p0, key):
        # p0 = P(cluster 0); <key>_home = which cluster is the home team
        return p0 if int(self.a[key + "_home"]) == 0 else 1.0 - p0

    def predict(self, hist, rn, sg):
        n = len(hist)
        out = {"t1": np.full(n, np.nan, np.float32), "t2": np.full(n, np.nan, np.float32),
               "t3": np.full(n, np.nan, np.float32)}
        if n == 0:
            return out
        out["t1"] = self._home(kmeans_p0(t1_features(hist, rn), self.a["t1_centers"]), "t1").astype(np.float32)
        out["t3"] = self._home(kmeans_p0(hist, self.a["t3_centers"]), "t3").astype(np.float32)
        if int(self.a.get("has_t2", 0)) and sg is not None:
            z = sg @ self.a["t2_w"] + float(self.a["t2_b"])
            p1 = 1.0 / (1.0 + np.exp(-z))          # P(cluster 1)
            out["t2"] = self._home(1.0 - p1, "t2").astype(np.float32)
        return out


def save_json(obj, path):
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2)


def atomic_parquet(df, path):
    tmp = path + ".tmp"
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)


In [ ]:
%%writefile det_worker.py
"""GPU detection worker: python det_worker.py <config.json> <worker_id> <n_workers>

Each worker takes the chunks whose index % n_workers == worker_id, runs the player model on every
frame and, every TEAM_EVERY-th frame, scores each player's torso with the frozen team models.
One Parquet file per chunk; chunks already on disk are skipped, so a rerun resumes.
"""
import json
import os
import sys
import time

import cv2
import numpy as np
import pandas as pd
import torch

from common import ROLE_PLAYER, Embedder, TeamModels, atomic_parquet, hsv_hist, role_map_from_names, torso_crop

cfg = json.load(open(sys.argv[1]))
wid, nw = int(sys.argv[2]), int(sys.argv[3])
cv2.setNumThreads(2)
torch.set_num_threads(2)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
half = device.startswith("cuda")

from ultralytics import YOLO  # noqa: E402

model = YOLO(cfg["model_path"])
roles = role_map_from_names(model.names)
keep_classes = sorted(roles)
emb = Embedder(device, use_siglip=cfg["use_siglip"])
tm = TeamModels.load(cfg["team_models_path"])

chunks = [c for i, c in enumerate(json.load(open(cfg["chunks_path"]))) if i % nw == wid]
print(f"[w{wid}] device={device} chunks={len(chunks)} classes={ {k: model.names[k] for k in keep_classes} }", flush=True)


def process(start, end):
    cap = cv2.VideoCapture(cfg["video"])
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    pos = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES)))
    if pos != start:
        raise RuntimeError(f"seek to {start} landed on {pos}")
    cols = {k: [] for k in ("frame", "x1", "y1", "x2", "y2", "conf", "role", "t1", "t2", "t3")}
    f = start
    while f <= end:
        frames, idx = [], []
        while len(frames) < cfg["batch"] and f <= end:
            ok, fr = cap.read()
            if not ok:
                break
            frames.append(fr)
            idx.append(f)
            f += 1
        if not frames:
            break
        results = model.predict(frames, imgsz=cfg["imgsz"], conf=cfg["det_conf"], classes=keep_classes,
                                agnostic_nms=True, max_det=60, half=half, device=0 if half else "cpu",
                                verbose=False)
        crops, ref = [], []
        for fr, fi, r in zip(frames, idx, results):
            if r.boxes is None or len(r.boxes) == 0:
                continue
            xyxy = r.boxes.xyxy.cpu().numpy()
            conf = r.boxes.conf.cpu().numpy()
            cls = r.boxes.cls.cpu().numpy().astype(int)
            score_team = fi % cfg["team_every"] == 0
            for j in range(len(xyxy)):
                role = roles.get(int(cls[j]))
                if role is None:
                    continue
                k = len(cols["frame"])
                cols["frame"].append(fi)
                cols["x1"].append(xyxy[j, 0]); cols["y1"].append(xyxy[j, 1])
                cols["x2"].append(xyxy[j, 2]); cols["y2"].append(xyxy[j, 3])
                cols["conf"].append(conf[j]); cols["role"].append(role)
                cols["t1"].append(np.nan); cols["t2"].append(np.nan); cols["t3"].append(np.nan)
                if score_team and role == ROLE_PLAYER:
                    c = torso_crop(fr, xyxy[j])
                    if c is not None:
                        crops.append(c)
                        ref.append(k)
        if crops:
            hist = np.stack([hsv_hist(c) for c in crops])
            rn, sg = emb.embed(crops)
            p = tm.predict(hist, rn, sg)
            for name in ("t1", "t2", "t3"):
                for i, k in enumerate(ref):
                    cols[name][k] = float(p[name][i])
    cap.release()
    if f <= end:
        raise RuntimeError(f"video ended at frame {f}, chunk wanted up to {end}")
    df = pd.DataFrame(cols)
    for c in ("x1", "y1", "x2", "y2", "conf", "t1", "t2", "t3"):
        df[c] = df[c].astype(np.float32)
    df["frame"] = df["frame"].astype(np.int32)
    df["role"] = df["role"].astype(np.int8)
    return df


for start, end in chunks:
    out = os.path.join(cfg["raw_dir"], f"chunk_{start:06d}_{end:06d}.parquet")
    if os.path.exists(out):
        continue
    t0 = time.time()
    df = process(start, end)
    atomic_parquet(df, out)
    n = end - start + 1
    print(f"[w{wid}] chunk {start}-{end}: {len(df)} dets, {n / (time.time() - t0):.1f} fps", flush=True)
print(f"[w{wid}] done", flush=True)


## Inputs, frame ranges and resume

In [ ]:
def find_one(name, required=True):
    hits = sorted(glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True))
    if not hits and required:
        raise FileNotFoundError(f"{name} not found under {DATA_DIR} — attach the dataset that holds it.")
    return hits[0] if hits else None

VIDEO = find_one("BAR-ATH_tactical.mp4")
ATD_TXT = find_one("BAR-ATH_tactical_atd.txt")
ATD_XML = find_one("BAR-ATH_tactical_atd.xml")
HOMO_CSV = find_one("BAR-ATH_tactical_homography.csv")
GT_PUBLIC = find_one("BAR-ATH_possession_gt_public.csv", required=False)

if MODEL_PATH is None:
    pts = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.pt"), recursive=True))
    pref = [p for p in pts if "supa-strikers-models" in p] or pts
    if not pref:
        raise FileNotFoundError("No .pt model found — attach the supa-strikers-models dataset.")
    MODEL_PATH = pref[0]
    if len(pref) > 1:
        print("Several .pt files found, using the first; set MODEL_PATH to pick another:", pref)

RAW_DIR = os.path.join(WORK, f"raw_dets_{RUN_MODE}")
CALIB_DIR = os.path.join(WORK, "calib")
TRACK_DIR = os.path.join(WORK, "tracks")
EVAL_DIR = os.path.join(WORK, "eval")
CLIP_DIR = os.path.join(WORK, "clips")
LOG_DIR = os.path.join(WORK, "logs")
for d in (RAW_DIR, CALIB_DIR, TRACK_DIR, EVAL_DIR, CLIP_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

# Resume across sessions: chunks from an attached supa-strikers-outputs dataset are copied in
# and skipped by the workers.
copied = 0
for p in glob.glob(os.path.join(DATA_DIR, "**", f"raw_dets_{RUN_MODE}", "*.parquet"), recursive=True):
    dst = os.path.join(RAW_DIR, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst)
        copied += 1
prev_calib = find_one("team_models.npz", required=False)
if prev_calib and not os.path.exists(os.path.join(CALIB_DIR, "team_models.npz")):
    for p in glob.glob(os.path.join(os.path.dirname(prev_calib), "*")):
        shutil.copy(p, CALIB_DIR)
    print("Restored calibration from", os.path.dirname(prev_calib))

# Frames processed (detected) and frames scored (evaluated)
RANGES = []                                   # (half, first, last) scored frames
for h, s, e in HALVES:
    RANGES.append((h, s, e if RUN_MODE == "full" else min(e, s + SMOKE_FRAMES - 1)))
EVAL = np.zeros(N_VIDEO, bool)
HALF_OF = np.zeros(N_VIDEO, np.int8)
for h, s, e in RANGES:
    EVAL[s:e + 1] = True
    HALF_OF[s - WARMUP:e + 1] = h

# Calibration frames always span both FULL halves (lighting / shadows change over the match), in
# smoke mode too, so the frozen team models are the same for smoke and full runs.
CALIB_FRAMES = np.concatenate([np.linspace(s + 250, e - 250, CALIB_FRAMES_PER_HALF).astype(int)
                               for _, s, e in HALVES])

CHUNKS = []
for h, s, e in RANGES:
    for a in range(s - WARMUP, e + 1, CHUNK):
        CHUNKS.append([a, min(a + CHUNK - 1, e)])

print("model:", MODEL_PATH)
print("video:", VIDEO)
print(f"mode={RUN_MODE}  scored frames={EVAL.sum():,}  chunks={len(CHUNKS)}  resumed chunks={copied}")


## ATD, team map and homography

In [ ]:
import xml.etree.ElementTree as ET
from common import HOME, AWAY, NO_TEAM, ROLE_PLAYER, ROLE_GK, ROLE_REF, ROLE_NAME, TEAM_LETTER
L, W = PITCH

# ---- ATD metadata: slot -> team, home/away names, attack direction --------------------------
root = ET.parse(ATD_XML).getroot()
teams = [t for t in root.iter("Team")]
home_id, away_id = teams[0].get("id"), teams[1].get("id")       # first <Team> = home (FIFA/EPTS)
TEAM_NAMES = (teams[0].findtext("Name"), teams[1].findtext("Name"))
ATTACK_1ST = {}
for t, side in zip(teams[:2], (HOME, AWAY)):
    for pp in t.iter("ProviderParameter"):
        if pp.findtext("Name") == "attack_direction_first_half":
            ATTACK_1ST[side] = pp.findtext("Value")
player_team = {p.get("id"): p.get("teamId") for p in root.iter("Player")}
chan_player = {c.get("id"): c.get("playerId") for c in root.iter("PlayerChannel")}
spec = next(root.iter("DataFormatSpecification"))
slot_players = [chan_player[r.get("playerChannelId")] for r in spec.iter("PlayerChannelRef")
                if r.get("playerChannelId", "").endswith("_x")]
SLOT_TEAM = np.array([HOME if player_team.get(p) == home_id else AWAY if player_team.get(p) == away_id
                      else NO_TEAM for p in slot_players], np.int8)
N_SLOTS = len(SLOT_TEAM)
print(f"home={TEAM_NAMES[0]}  away={TEAM_NAMES[1]}  slots={N_SLOTS} "
      f"(H {np.sum(SLOT_TEAM == HOME)}, A {np.sum(SLOT_TEAM == AWAY)})  attack 1st half: {ATTACK_1ST}")

# ---- ATD positions (normalised 0-1) for the frames we score; video_frame = frame_id - 1 -------
t0 = time.time()
ATD = np.full((N_VIDEO, N_SLOTS, 2), np.nan, np.float32)
need = HALF_OF != 0
need[CALIB_FRAMES] = True
with open(ATD_TXT) as fh:
    for line in fh:
        fid, players, _ball = line.rstrip("\n").split(":")
        vf = int(fid) - 1
        if vf < 0 or vf >= N_VIDEO or not need[vf]:
            continue
        for i, p in enumerate(players.split(";")):
            if p[0] != "N":
                x, y, _ = p.split(",")
                ATD[vf, i, 0] = float(x)
                ATD[vf, i, 1] = float(y)
print(f"ATD parsed in {time.time() - t0:.0f}s; mean objects per scored frame: "
      f"{np.isfinite(ATD[EVAL, :, 0]).sum(1).mean():.1f}")

# ---- Homography: normalised pitch (0-1) -> pixels; inverse for pixels -> pitch --------------
hdf = pd.read_csv(HOMO_CSV)
H = hdf[[f"h{r}{c}" for r in range(3) for c in range(3)]].values.astype(np.float64).reshape(-1, 3, 3)
H_VALID = hdf["valid"].values.astype(bool)
H_INV = np.full_like(H, np.nan)
H_INV[H_VALID] = np.linalg.inv(H[H_VALID])
print(f"homography: {len(H):,} frames, valid {H_VALID.mean():.1%}, "
      f"median reprojection error in scored frames {np.nanmedian(hdf['reprojection_error'].values[EVAL]):.2f}")

def pix_to_pitch(frames, u, v):
    """Pixel feet points -> metres (x along the 105.3 m length, y across)."""
    Hi = H_INV[np.asarray(frames)]
    p = np.einsum("nij,nj->ni", Hi, np.stack([u, v, np.ones_like(u)], 1).astype(np.float64))
    return (p[:, 0] / p[:, 2] * L).astype(np.float32), (p[:, 1] / p[:, 2] * W).astype(np.float32)

def pitch_to_pix(frame, xn, yn):
    """Normalised pitch -> pixels for one frame."""
    p = H[frame] @ np.stack([xn, yn, np.ones_like(xn)]).astype(np.float64)
    return p[0] / p[2], p[1] / p[2]

def match(a, b, gate):
    """Hungarian match of two (n,2) point sets in metres; returns (ia, ib, dist) within gate."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0)
    d = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)
    ia, ib = linear_sum_assignment(np.where(d > gate, 1e6, d))
    ok = d[ia, ib] <= gate
    return ia[ok], ib[ok], d[ia, ib][ok]

def atd_frame_m(f):
    """ATD positions in metres + slot team for one frame (finite slots only)."""
    xy = ATD[f]
    ok = np.isfinite(xy[:, 0])
    return xy[ok] * np.array([L, W], np.float32), SLOT_TEAM[ok], np.flatnonzero(ok)


## Video sanity check

In [ ]:
cap = cv2.VideoCapture(VIDEO)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
vid_fps = cap.get(cv2.CAP_PROP_FPS)
VID_W, VID_H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"video: {n_frames:,} frames, {vid_fps:.2f} fps, {VID_W}x{VID_H}")
if abs(n_frames - N_VIDEO) > 2 or abs(vid_fps - FPS) > 0.1:
    print("WARNING: frame count / fps differ from DATA.md — check you attached the TACTICAL video.")

def read_at(f):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, fr = cap.read()
    return fr if ok else None

# Seeking must land on the exact frame: compare a direct seek with a seek 60 frames earlier
# followed by sequential reads. (Calibration then checks alignment against the ATD as well.)
bad = []
for f in (RANGES[0][1], (RANGES[0][1] + RANGES[0][2]) // 2 + 7, (RANGES[-1][1] + RANGES[-1][2]) // 2 + 13):
    direct = read_at(f)
    cap.set(cv2.CAP_PROP_POS_FRAMES, f - 60)
    for _ in range(60):
        cap.read()
    ok, seq = cap.read()
    if direct is None or seq is None:
        raise RuntimeError(f"could not read frame {f}")
    diff = float(np.abs(direct.astype(np.int16) - seq.astype(np.int16)).mean())
    print(f"  frame {f}: direct-seek vs sequential mean abs diff = {diff:.2f}")
    if diff > 1.0:
        bad.append(f)
cap.release()
if bad:
    raise RuntimeError(f"Seeking is not frame-accurate at {bad}. Re-encode the video with a keyframe "
                       "every 25 frames (ffmpeg -g 25) or ask for the PyAV fallback.")
print("seek check passed")


## Calibration — fit the frozen team models (T1, T2, T3)
~150 frames across both halves. Clusters are named Home/Away by matching our players to ATD
positions (ATD team membership), never by eye. **Check:** the ATD alignment line (median distance
should be well under 1 m) and the two crop grids (each should be one kit).

In [ ]:
from ultralytics import YOLO
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from common import (Embedder, TeamModels, role_map_from_names, torso_crop, hsv_hist, t1_features,
                    save_json)

TEAM_MODELS_PATH = os.path.join(CALIB_DIR, "team_models.npz")
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

det_model = YOLO(MODEL_PATH)
ROLES = role_map_from_names(det_model.names)
print("model classes:", det_model.names)
print("used as:", {det_model.names[k]: ROLE_NAME[v] for k, v in ROLES.items()})
if ROLE_REF not in ROLES.values():
    print("NOTE: no referee class — referees will fall into the player pool.")

FORCE_RECALIB = False   # True refits the team models: then also delete raw_dets_* (old team scores)


def calibrate():
    global det_fps
    # ---- detect on the calibration frames and time the model ------------------------------------
    cap = cv2.VideoCapture(VIDEO)
    frames, crops, crop_rows, det_rows = {}, [], [], []
    for f in CALIB_FRAMES:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f))
        ok, fr = cap.read()
        if ok:
            frames[int(f)] = fr
    cap.release()
    keys = list(frames)
    t0 = time.time()
    for i in range(0, len(keys), BATCH):
        batch = [frames[k] for k in keys[i:i + BATCH]]
        res = det_model.predict(batch, imgsz=IMGSZ, conf=DET_CONF, classes=sorted(ROLES), agnostic_nms=True,
                                max_det=60, half=DEVICE != "cpu", device=0 if DEVICE != "cpu" else "cpu",
                                verbose=False)
        for f, r in zip(keys[i:i + BATCH], res):
            for b, c, k in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()):
                role = ROLES.get(int(k))
                if role is None:
                    continue
                det_rows.append((f, *b, c, role))
                if role == ROLE_PLAYER and c >= 0.4:
                    crop = torso_crop(frames[f], b)
                    if crop is not None:
                        crops.append(crop)
                        crop_rows.append(len(det_rows) - 1)
    det_fps = len(keys) / (time.time() - t0)
    cal = pd.DataFrame(det_rows, columns=["frame", "x1", "y1", "x2", "y2", "conf", "role"])
    print(f"{len(keys)} frames -> {len(cal)} detections ({len(cal) / max(len(keys), 1):.1f}/frame), "
          f"{len(crops)} player torso crops; model speed ≈ {det_fps:.1f} frames/s per GPU (incl. warm-up)")
    print("detections per role per frame:", (cal.groupby("role").size() / len(keys)).rename(ROLE_NAME).round(1).to_dict())

    # ---- alignment check against the ATD: do our feet land on ATD positions? ---------------------
    cal["x_m"], cal["y_m"] = pix_to_pitch(cal.frame.values, ((cal.x1 + cal.x2) / 2).values, cal.y2.values)
    cal["atd_team"] = NO_TEAM
    dists = []
    for f, g in cal[cal.role != ROLE_REF].groupby("frame"):
        axy, ateam, _ = atd_frame_m(f)
        ia, ib, d = match(g[["x_m", "y_m"]].values, axy, gate=2.5)
        cal.loc[g.index[ia], "atd_team"] = ateam[ib]
        dists.append(d)
    dists = np.concatenate(dists) if dists else np.zeros(0)
    n_cmp = (cal.role != ROLE_REF).sum()
    print(f"ATD alignment: {len(dists) / max(n_cmp, 1):.0%} of our player/GK boxes have an ATD object within "
          f"2.5 m, median distance {np.median(dists) if len(dists) else float('nan'):.2f} m")
    if len(dists) == 0 or np.median(dists) > 1.5:
        print("WARNING: weak alignment — check the video is the tactical one and frames are not offset.")

    # ---- fit the frozen team models --------------------------------------------------------------
    emb = Embedder(DEVICE, use_siglip=USE_SIGLIP)
    hist = np.stack([hsv_hist(c) for c in crops])
    rn, sg = emb.embed(crops)
    atd_team = cal.loc[crop_rows, "atd_team"].values
    arrays, report = {}, {}

    def home_cluster(labels, key):
        """Which cluster is home: the one whose matched crops are most often ATD-home."""
        m = atd_team >= 0
        frac = [np.mean(atd_team[m & (labels == c)] == HOME) if np.any(m & (labels == c)) else 0.5 for c in (0, 1)]
        hc = int(np.argmax(frac))
        pred_home = labels == hc
        agree = np.mean(pred_home[m] == (atd_team[m] == HOME)) if m.any() else float("nan")
        report[key] = {"home_cluster": hc, "cluster_sizes": np.bincount(labels, minlength=2).tolist(),
                       "agreement_with_atd": round(float(agree), 3), "frac_home_by_cluster": [round(float(x), 3) for x in frac]}
        return hc

    km1 = KMeans(2, random_state=42, n_init=10).fit(t1_features(hist, rn))
    arrays["t1_centers"], arrays["t1_home"] = km1.cluster_centers_.astype(np.float32), home_cluster(km1.labels_, "T1")
    km3 = KMeans(2, random_state=42, n_init=10).fit(hist)
    arrays["t3_centers"], arrays["t3_home"] = km3.cluster_centers_.astype(np.float32), home_cluster(km3.labels_, "T3")
    arrays["has_t2"] = 0
    if sg is not None:
        import umap
        red = umap.UMAP(n_components=3, random_state=42).fit_transform(sg)
        km2 = KMeans(2, random_state=42, n_init=10).fit(red)
        lr = LogisticRegression(max_iter=5000).fit(sg, km2.labels_)   # fast stand-in for umap.transform
        arrays.update(has_t2=1, t2_w=lr.coef_[0].astype(np.float32), t2_b=np.float32(lr.intercept_[0]),
                      t2_home=home_cluster(km2.labels_, "T2"))
        report["T2"]["lr_fit_accuracy"] = round(float(lr.score(sg, km2.labels_)), 3)
    TeamModels(arrays).save(TEAM_MODELS_PATH)
    report.update(det_fps=round(det_fps, 1), n_crops=len(crops), atd_match_median_m=float(np.median(dists)) if len(dists) else None)
    save_json(report, os.path.join(CALIB_DIR, "calibration_report.json"))
    print(json.dumps({k: v for k, v in report.items() if k.startswith("T")}, indent=1))

    # ---- review images: crops per cluster (T1) ---------------------------------------------------
    def crop_grid(idx, title, path, n=48, cell=64):
        tiles = [cv2.resize(crops[i], (cell, cell)) for i in idx[:n]]
        if not tiles:
            return
        tiles += [np.full((cell, cell, 3), 255, np.uint8)] * (-len(tiles) % 12)
        grid = np.vstack([np.hstack(tiles[r:r + 12]) for r in range(0, len(tiles), 12)])
        plt.figure(figsize=(9, grid.shape[0] / grid.shape[1] * 9 + 0.4))
        plt.imshow(cv2.cvtColor(grid, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title(title, loc="left", fontsize=10)
        plt.savefig(path, dpi=110, bbox_inches="tight"); plt.show()

    for c in (0, 1):
        side = "HOME" if c == arrays["t1_home"] else "AWAY"
        crop_grid(np.flatnonzero(km1.labels_ == c), f"T1 cluster {c} -> {side} ({TEAM_NAMES[0 if side == 'HOME' else 1]})",
                  os.path.join(CALIB_DIR, f"t1_cluster{c}.png"))

    # ---- runtime projection ----------------------------------------------------------------------
    n_det_frames = sum(e - s + 1 for s, e in CHUNKS)
    n_workers = max(1, torch.cuda.device_count())
    est_h = n_det_frames / (det_fps * 0.7 * n_workers) / 3600     # 0.7: decoding + team scoring overhead
    print(f"Projected detection pass: {n_det_frames:,} frames on {n_workers} worker(s) ≈ {est_h:.1f} h")
    del emb
    gc.collect(); torch.cuda.empty_cache()


det_fps = None
if os.path.exists(TEAM_MODELS_PATH) and not FORCE_RECALIB:
    print("Using existing frozen team models:", TEAM_MODELS_PATH)
    rep = os.path.join(CALIB_DIR, "calibration_report.json")
    if os.path.exists(rep):
        print(open(rep).read())
else:
    calibrate()
del det_model
gc.collect(); torch.cuda.empty_cache()


## Detection pass (both GPUs, resumable)
One worker per GPU; one Parquet per chunk. If the session dies, rerun this cell (or attach this
run's output as `supa-strikers-outputs` in a new session): finished chunks are skipped.

In [ ]:
import subprocess

def chunk_path(s, e):
    return os.path.join(RAW_DIR, f"chunk_{s:06d}_{e:06d}.parquet")

todo = [c for c in CHUNKS if not os.path.exists(chunk_path(*c))]
print(f"{len(CHUNKS) - len(todo)}/{len(CHUNKS)} chunks already on disk, {len(todo)} to run")

if todo:
    cfg_w = dict(video=VIDEO, model_path=os.path.abspath(MODEL_PATH), team_models_path=TEAM_MODELS_PATH,
                 chunks_path=os.path.join(WORK, "chunks_todo.json"), raw_dir=RAW_DIR, imgsz=IMGSZ,
                 batch=BATCH, det_conf=DET_CONF, team_every=TEAM_EVERY, use_siglip=USE_SIGLIP)
    save_json(todo, cfg_w["chunks_path"])
    save_json(cfg_w, os.path.join(WORK, "worker_cfg.json"))
    n_workers = max(1, torch.cuda.device_count())
    procs, logs = [], []
    for w in range(n_workers):
        env = dict(os.environ)
        if torch.cuda.is_available():
            env["CUDA_VISIBLE_DEVICES"] = str(w)
        logs.append(os.path.join(LOG_DIR, f"worker{w}.log"))
        procs.append(subprocess.Popen([sys.executable, os.path.join(WORK, "det_worker.py"),
                                       os.path.join(WORK, "worker_cfg.json"), str(w), str(n_workers)],
                                      env=env, cwd=WORK, stdout=open(logs[-1], "w"), stderr=subprocess.STDOUT))
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(60 if RUN_MODE == "full" else 15)
        done = sum(os.path.exists(chunk_path(*c)) for c in CHUNKS)
        el = (time.time() - t0) / 60
        left = len(CHUNKS) - done
        fin = len(todo) - left
        eta = el / fin * left if fin else float("nan")
        print(f"[{el:6.1f} min] chunks {done}/{len(CHUNKS)}  ETA ≈ {eta:.0f} min", flush=True)
    for w, p in enumerate(procs):
        if p.returncode != 0:
            print(open(logs[w]).read()[-3000:])
            raise RuntimeError(f"worker {w} failed (exit {p.returncode}); its log is above. Rerun this cell to resume.")
    print(f"detection pass finished in {(time.time() - t0) / 60:.1f} min")
missing = [c for c in CHUNKS if not os.path.exists(chunk_path(*c))]
assert not missing, f"missing chunks: {missing[:5]}"


## Load detections and project feet to pitch metres

In [ ]:
D = pd.concat([pd.read_parquet(chunk_path(*c)) for c in CHUNKS], ignore_index=True)
D = D[HALF_OF[D.frame.values] != 0].sort_values(["frame", "conf"], ascending=[True, False]).reset_index(drop=True)
D["half"] = HALF_OF[D.frame.values]
D["fx"] = ((D.x1 + D.x2) / 2).astype(np.float32)        # feet point in pixels
D["fy"] = D.y2.astype(np.float32)
D["x_m"], D["y_m"] = pix_to_pitch(D.frame.values, D.fx.values, D.fy.values)
print(f"{len(D):,} detections, {len(D) / (HALF_OF != 0).sum():.1f} per frame; by role:",
      D.role.map(ROLE_NAME).value_counts().to_dict())
print(f"team scores present on {np.isfinite(D.t1).mean():.0%} of rows (players on every {TEAM_EVERY}rd frame)")


## Tracking — K1 (old ByteTrack), K2 (tuned ByteTrack), K3 (pitch-metre tracker)
All CPU, on the cached detections.

In [ ]:
import supervision as sv

FR = D.frame.values
XYXY = D[["x1", "y1", "x2", "y2"]].values.astype(np.float32)
CONF = D.conf.values.astype(np.float32)
XM = np.stack([D.x_m.values, D.y_m.values], 1).astype(np.float64)
TRACK_RANGES = [(h, s - WARMUP, e) for h, s, e in RANGES]


def run_bytetrack(conf_min, **kw):
    """Image-space ByteTrack (supervision). Returns a track id per detection (-1 = untracked)."""
    ids = np.full(len(D), -1, np.int64)
    for h, f0, f1 in TRACK_RANGES:
        tr = sv.ByteTrack(**kw)
        b = np.searchsorted(FR, np.arange(f0, f1 + 2))
        for i in range(f1 - f0 + 1):
            idx = np.arange(b[i], b[i + 1])
            idx = idx[CONF[idx] >= conf_min]
            det = (sv.Detections(xyxy=XYXY[idx], confidence=CONF[idx], class_id=np.zeros(len(idx), int),
                                 data={"det_idx": idx}) if len(idx) else sv.Detections.empty())
            out = tr.update_with_detections(det)
            if len(out):
                ok = out.tracker_id >= 0
                ids[out.data["det_idx"][ok]] = out.tracker_id[ok] + h * 1_000_000
    return ids


def run_pitch_tracker(hi=0.4, lo=0.1, gate=2.5, gate_lo=1.5, max_lost=100, alpha=0.8, beta=0.25, vmax=10.0):
    """Tracker in pitch metres (homography removes camera pan/zoom). Alpha-beta filter + two-stage
    Hungarian matching like ByteTrack: confident boxes first, then low-confidence boxes for the
    tracks still unmatched."""
    ids = np.full(len(D), -1, np.int64)
    for h, f0, f1 in TRACK_RANGES:
        pos = np.zeros((0, 2)); vel = np.zeros((0, 2)); last = np.zeros(0, int); tid = np.zeros(0, np.int64)
        nxt = 1
        b = np.searchsorted(FR, np.arange(f0, f1 + 2))
        for i, f in enumerate(range(f0, f1 + 1)):
            alive = (f - last) <= max_lost
            pos, vel, last, tid = pos[alive], vel[alive], last[alive], tid[alive]
            dt = (f - last) / FPS
            pred = pos + vel * dt[:, None]
            idx = np.arange(b[i], b[i + 1])
            idx = idx[(CONF[idx] >= lo) & np.isfinite(XM[idx, 0])]
            z = XM[idx]
            used_t = np.zeros(len(pos), bool); used_d = np.zeros(len(idx), bool)
            for stage in (0, 1):
                dmask = (CONF[idx] >= hi) if stage == 0 else (CONF[idx] < hi)
                dmask &= ~used_d
                tmask = ~used_t if stage == 0 else (~used_t & ((f - last) <= 3))
                ti, di = np.flatnonzero(tmask), np.flatnonzero(dmask)
                if len(ti) == 0 or len(di) == 0:
                    continue
                g = (np.minimum(gate + 0.15 * (f - last[ti] - 1), 6.0) if stage == 0 else np.full(len(ti), gate_lo))
                d = np.linalg.norm(pred[ti][:, None, :] - z[di][None, :, :], axis=2)
                r, c = linear_sum_assignment(np.where(d > g[:, None], 1e6, d))
                ok = d[r, c] <= g[r]
                r, c = ti[r[ok]], di[c[ok]]
                innov = z[c] - pred[r]
                pos[r] = pred[r] + alpha * innov
                vel[r] = vel[r] + beta * innov / np.maximum(dt[r], 1 / FPS)[:, None]
                sp = np.linalg.norm(vel[r], axis=1, keepdims=True)
                vel[r] *= np.minimum(1.0, vmax / np.maximum(sp, 1e-9))
                last[r] = f
                used_t[r] = True; used_d[c] = True
                ids[idx[c]] = tid[r]
            new = np.flatnonzero(~used_d & (CONF[idx] >= hi))
            if len(new):
                new_ids = h * 1_000_000 + np.arange(nxt, nxt + len(new)); nxt += len(new)
                pos = np.vstack([pos, z[new]]); vel = np.vstack([vel, np.zeros((len(new), 2))])
                last = np.concatenate([last, np.full(len(new), f)]); tid = np.concatenate([tid, new_ids])
                ids[idx[new]] = new_ids
    return ids


TRACKERS = {
    "K1": lambda: run_bytetrack(0.30),                                          # old script
    "K2": lambda: run_bytetrack(DET_CONF, track_activation_threshold=0.4, lost_track_buffer=120,
                                minimum_matching_threshold=0.8, frame_rate=FPS),  # tuned
    "K3": lambda: run_pitch_tracker(),                                          # pitch metres
}
TRACK_IDS = {}
for k, fn in TRACKERS.items():
    t0 = time.time()
    TRACK_IDS[k] = fn()
    ids = TRACK_IDS[k]
    print(f"{k}: {np.mean(ids >= 0):.1%} of detections tracked, {len(np.unique(ids[ids >= 0])):,} raw IDs, "
          f"{time.time() - t0:.0f}s")


## Team and role per track, goalkeeper side rule, stitching and smoothing

In [ ]:
def assign_tracks(ids, tkey):
    """Per-track votes: role (confidence-weighted), team (mean P(home)); goalkeepers by side."""
    keep = ids >= 0
    T = D.loc[keep, ["frame", "half", "x1", "y1", "x2", "y2", "conf", "role", "x_m", "y_m"]].copy()
    T["id"] = ids[keep]
    T["p"] = D.loc[keep, tkey].values
    T = T.sort_values(["id", "frame", "conf"], ascending=[True, True, False])
    T = T.drop_duplicates(["id", "frame"])                    # one box per track per frame
    n = T.groupby("id").size()
    T = T[T.id.isin(n.index[n >= MIN_TRACK_DETS])]
    # role vote
    w = T.groupby(["id", "role"]).conf.sum().unstack(fill_value=0)
    T["role"] = T.id.map(w.idxmax(axis=1)).astype(np.int8)
    # team vote for players
    pm = T.groupby("id").p.mean()
    team = np.where(pm.isna(), NO_TEAM, np.where(pm >= 0.5, HOME, AWAY))
    T["team"] = T.id.map(pd.Series(team, index=pm.index)).astype(np.int8)
    T.loc[T.role != ROLE_PLAYER, "team"] = NO_TEAM
    T["team"] = side_rule(T)
    return T.sort_values(["frame", "id"]).reset_index(drop=True)


LEFT_TEAM = {}   # per variant run, filled by side_rule (half -> team defending x≈0)

def side_rule(T):
    """At each kickoff (first 10 s) the team whose players stand on the left defends the left goal;
    a goalkeeper on the left half belongs to that team."""
    team = T.team.values.copy()
    for h, s, e in RANGES:
        k = T[(T.frame >= s) & (T.frame < s + 10 * FPS) & (T.role == ROLE_PLAYER) & (T.team >= 0)]
        mx = k.groupby("team").x_m.mean()
        if len(mx) < 2:
            continue
        left = int(mx.idxmin())
        LEFT_TEAM[h] = left
        g = T[(T.half == h) & (T.role == ROLE_GK)]
        gx = g.groupby("id").x_m.median()
        gk_team = pd.Series(np.where(gx < L / 2, left, 1 - left), index=gx.index)
        sel = (T.half.values == h) & (T.role.values == ROLE_GK)
        team[sel] = T.id[sel].map(gk_team).values
    return team.astype(np.int8)


def stitch(T, max_gap=STITCH_MAX_GAP, vmax=STITCH_MAX_SPEED, base=1.5):
    """Join a track that ends to one that starts shortly after, nearby, same role and team."""
    g = T.groupby("id")
    E = g.tail(1)[["id", "frame", "x_m", "y_m", "role", "team", "half"]].to_numpy()
    S = g.head(1)[["id", "frame", "x_m", "y_m", "role", "team", "half"]].to_numpy()
    S = S[np.argsort(S[:, 1], kind="stable")]
    cand = []
    for e in E:
        lo = np.searchsorted(S[:, 1], e[1] + 1)
        hi = np.searchsorted(S[:, 1], e[1] + max_gap, side="right")
        s = S[lo:hi]
        if not len(s):
            continue
        s = s[(s[:, 0] != e[0]) & (s[:, 4] == e[4]) & (s[:, 5] == e[5]) & (s[:, 6] == e[6])]
        gap = (s[:, 1] - e[1]) / FPS
        d = np.hypot(s[:, 2] - e[2], s[:, 3] - e[3])
        ok = np.isfinite(d) & (d <= base + vmax * gap)
        cand += [(dd + 2.0 * gg, int(e[0]), int(b)) for dd, gg, b in zip(d[ok], gap[ok], s[ok, 0])]
    cand.sort()
    parent, used_end, used_start = {}, set(), set()
    for _, a, b in cand:
        if a in used_end or b in used_start:
            continue
        used_end.add(a); used_start.add(b); parent[b] = a

    def root(x):
        while x in parent:
            x = parent[x]
        return x
    T = T.copy()
    if parent:
        mp = {b: root(b) for b in parent}
        T["id"] = T.id.map(lambda x: mp.get(x, x)).astype(np.int64)
    return T, len(parent)


def smooth(T, win=SMOOTH_WINDOW):
    T = T.sort_values(["id", "frame"]).copy()
    x, y = T.x_m.values.astype(np.float64), T.y_m.values.astype(np.float64)
    new_seg = np.r_[True, (T.id.values[1:] != T.id.values[:-1]) | (np.diff(T.frame.values) != 1)]
    seg = np.cumsum(new_seg)
    bounds = np.r_[np.flatnonzero(new_seg), len(T)]
    for a, b in zip(bounds[:-1], bounds[1:]):
        if b - a >= win and np.all(np.isfinite(x[a:b])):
            x[a:b] = savgol_filter(x[a:b], win, 2)
            y[a:b] = savgol_filter(y[a:b], win, 2)
    T["x_m"], T["y_m"] = x.astype(np.float32), y.astype(np.float32)
    return T.sort_values(["frame", "id"]).reset_index(drop=True)


TEAM_KEYS = [k for k in ("t1", "t2", "t3") if np.isfinite(D[k]).any()]
print("team variants available:", TEAM_KEYS)


## Scorecard — ATD vs every variant (label-free metrics)

In [ ]:
EVAL_FRAMES = np.flatnonzero(EVAL)


def kickoff_left(T, h, s):
    k = T[(T.frame >= s) & (T.frame < s + 10 * FPS) & (T.role != ROLE_REF) & (T.team >= 0)]
    mx = k.groupby("team").x_m.mean()
    return int(mx.idxmin()) if len(mx) == 2 else None


def segments(T, split_gap=FPS):
    """Continuous pieces of each id (a gap > 1 s starts a new piece): comparable across ATD slots
    (reused by different people) and our IDs."""
    T = T.sort_values(["id", "frame"])
    new = np.r_[True, (T.id.values[1:] != T.id.values[:-1]) | (np.diff(T.frame.values) > split_gap)]
    seg = np.cumsum(new)
    return T.assign(seg=seg)


def scorecard(T, name, has_gk=True, has_p=True):
    T = T[EVAL[T.frame.values]]
    P = T[T.role != ROLE_REF]
    nE = len(EVAL_FRAMES)
    cnt = np.bincount(P.frame.values, minlength=N_VIDEO)[EVAL_FRAMES]
    cH = np.bincount(P.frame.values[P.team.values == HOME], minlength=N_VIDEO)[EVAL_FRAMES]
    cA = np.bincount(P.frame.values[P.team.values == AWAY], minlength=N_VIDEO)[EVAL_FRAMES]
    r = {"source": name, "people_per_frame": cnt.mean(),
         "coverage_20_23": np.mean((cnt >= 20) & (cnt <= 23)),
         "team_overflow": np.mean((cH > 11) | (cA > 11)),
         "unknown_team_share": np.mean(P.team.values == NO_TEAM) if len(P) else np.nan}
    if has_gk:
        G = T[T.role == ROLE_GK]
        ok_side = np.zeros(len(G), bool)
        for h, s, e in RANGES:
            left = kickoff_left(T, h, s)
            if left is None:
                continue
            m = G.half.values == h
            own_left = G.team.values[m] == left
            ok_side[m] = np.where(own_left, G.x_m.values[m] < L / 2, G.x_m.values[m] >= L / 2)
        gH = np.bincount(G.frame.values[(G.team.values == HOME) & ok_side], minlength=N_VIDEO)[EVAL_FRAMES]
        gA = np.bincount(G.frame.values[(G.team.values == AWAY) & ok_side], minlength=N_VIDEO)[EVAL_FRAMES]
        gall = np.bincount(G.frame.values, minlength=N_VIDEO)[EVAL_FRAMES]
        r["gk_sane"] = np.mean((gH == 1) & (gA == 1) & (gall == 2))
        R = T[T.role == ROLE_REF]
        r["referees_per_frame"] = np.bincount(R.frame.values, minlength=N_VIDEO)[EVAL_FRAMES].mean()
    # Jumps over a 0.2 s lag (5 frames): robust to per-frame jitter, still catches ID swaps.
    S = P.sort_values(["id", "frame"])
    lag = 5
    iv, fv = S.id.values, S.frame.values
    same = (iv[lag:] == iv[:-lag]) & (fv[lag:] - fv[:-lag] == lag)
    sp = np.hypot(S.x_m.values[lag:] - S.x_m.values[:-lag], S.y_m.values[lag:] - S.y_m.values[:-lag])[same] * FPS / lag
    r["jumps_per_1000"] = 1000 * np.mean(sp > JUMP_SPEED) if len(sp) else np.nan
    sg = segments(P)
    seg_len = sg.groupby("seg").size() / FPS
    r["segments_per_half"] = sg.seg.nunique() / len(RANGES)
    r["median_segment_s"] = float(seg_len.median()) if len(seg_len) else np.nan
    if has_p:
        Q = P[np.isfinite(P.p.values) & (P.team.values >= 0) & (P.role.values == ROLE_PLAYER)]
        r["team_flip_rate"] = np.mean((Q.p.values >= 0.5) != (Q.team.values == HOME)) if len(Q) else np.nan
    r["off_pitch"] = np.mean((P.x_m < -2) | (P.x_m > L + 2) | (P.y_m < -2) | (P.y_m > W + 2))
    return r


# ---- ATD as a track table ----------------------------------------------------------------------
fi, si = np.nonzero(np.isfinite(ATD[:, :, 0]) & EVAL[:, None])
ATD_T = pd.DataFrame({"frame": fi.astype(np.int32), "id": si + 1, "half": HALF_OF[fi],
                      "role": np.int8(ROLE_PLAYER), "team": SLOT_TEAM[si],
                      "x_m": ATD[fi, si, 0] * L, "y_m": ATD[fi, si, 1] * W})

# ---- build every variant and score it ----------------------------------------------------------
rows = [scorecard(ATD_T, "ATD", has_gk=False, has_p=False)]
VARIANTS = {}
for k in TRACK_IDS:
    for t in TEAM_KEYS:
        t0 = time.time()
        T = assign_tracks(TRACK_IDS[k], t)
        name = f"{k}-{t.upper()}"
        rows.append(scorecard(T, name))
        Tp, n_joined = stitch(T)
        Tp = smooth(Tp)
        rows.append(scorecard(Tp, name + "+post"))
        VARIANTS[name + "+post"] = Tp
        if SAVE_ALL_VARIANTS:
            Tp.to_parquet(os.path.join(TRACK_DIR, f"{name}+post.parquet"), index=False)
        left = {h: kickoff_left(Tp, h, s) for h, s, e in RANGES}
        print(f"{name}: {n_joined} joins, left-defending team per half {left} "
              f"(ATD meta: home attacks {ATTACK_1ST.get(HOME)} in 1st half), {time.time() - t0:.0f}s")
SCORE = pd.DataFrame(rows).set_index("source")


## Head to head with ATD, best variant, per-minute fault map

In [ ]:
H2H_STEP = 5   # every 5th scored frame


def head_to_head(T):
    """Per sampled frame: match ATD to our players/GKs (2 m gate) and to our referees (1 m)."""
    P = T[(T.role != ROLE_REF) & EVAL[T.frame.values]]
    R = T[(T.role == ROLE_REF) & EVAL[T.frame.values]]
    pb = np.searchsorted(P.frame.values, np.arange(N_VIDEO + 1))
    rb = np.searchsorted(R.frame.values, np.arange(N_VIDEO + 1))
    pxy, pteam = P[["x_m", "y_m"]].values, P.team.values
    rxy = R[["x_m", "y_m"]].values
    out = []
    for f in EVAL_FRAMES[::H2H_STEP]:
        axy, ateam, _ = atd_frame_m(f)
        ours = pxy[pb[f]:pb[f + 1]]
        ia, ib, d = match(axy, ours, 2.0)
        tm = pteam[pb[f]:pb[f + 1]][ib]
        known = tm >= 0
        free = np.setdiff1d(np.arange(len(axy)), ia)          # ATD objects with no player of ours
        ja, jb, _ = match(axy[free], rxy[rb[f]:rb[f + 1]], 1.0)  # ...sitting on one of our referees
        out.append((f, len(axy), len(ours), len(ia), np.median(d) if len(d) else np.nan,
                    np.sum(tm[known] == ateam[ia][known]), known.sum(), len(ja)))
    return pd.DataFrame(out, columns=["frame", "n_atd", "n_ours", "matched", "med_dist",
                                      "team_same", "team_known", "atd_on_referee"])


H2H = {}
for name, T in VARIANTS.items():
    h = head_to_head(T)
    H2H[name] = h
    SCORE.loc[name, "h2h_match_rate"] = h.matched.sum() / max(h.n_atd.sum(), 1)
    SCORE.loc[name, "h2h_median_dist_m"] = h.med_dist.median()
    SCORE.loc[name, "h2h_team_agreement"] = h.team_same.sum() / max(h.team_known.sum(), 1)
    SCORE.loc[name, "atd_slots_on_referee_per_frame"] = h.atd_on_referee.mean()

# ---- pick the best variant ----------------------------------------------------------------------
ours = SCORE.index[SCORE.index.str.endswith("+post")]
atd = SCORE.loc["ATD"]
SCORE["beats_atd"] = False
SCORE.loc[ours, "beats_atd"] = ((SCORE.loc[ours, "team_overflow"] < atd.team_overflow)
                                & (SCORE.loc[ours, "jumps_per_1000"] < atd.jumps_per_1000)
                                & (SCORE.loc[ours, "coverage_20_23"] > atd.coverage_20_23))
rank = (SCORE.loc[ours, "team_overflow"].rank() + SCORE.loc[ours, "jumps_per_1000"].rank()
        + (-SCORE.loc[ours, "coverage_20_23"]).rank() + SCORE.loc[ours, "segments_per_half"].rank()
        + (-SCORE.loc[ours, "gk_sane"]).rank() + SCORE.loc[ours, "team_flip_rate"].rank())
SCORE.loc[ours, "rank_score"] = rank
pool = SCORE.loc[ours][SCORE.loc[ours, "beats_atd"]]
BEST = BEST_VARIANT or (pool if len(pool) else SCORE.loc[ours]).rank_score.idxmin()
print("Best variant:", BEST, "| beats ATD on overflow + jumps + coverage:", bool(SCORE.loc[BEST, "beats_atd"]))

SCORE.round(4).to_csv(os.path.join(EVAL_DIR, "scorecard.csv"))
show = ["people_per_frame", "coverage_20_23", "team_overflow", "gk_sane", "referees_per_frame",
        "jumps_per_1000", "segments_per_half", "median_segment_s", "team_flip_rate", "off_pitch",
        "h2h_match_rate", "h2h_team_agreement", "atd_slots_on_referee_per_frame", "beats_atd"]
try:
    display(SCORE[show].round(3))          # notebook
except NameError:
    print(SCORE[show].round(3).to_string())

# ---- fault map per minute: where ATD and/or ours look wrong ------------------------------------
def per_minute(T):
    T = T[EVAL[T.frame.values]]
    P = T[T.role != ROLE_REF]
    m = pd.DataFrame({"frame": EVAL_FRAMES})
    m["cnt"] = np.bincount(P.frame.values, minlength=N_VIDEO)[EVAL_FRAMES]
    cH = np.bincount(P.frame.values[P.team.values == HOME], minlength=N_VIDEO)[EVAL_FRAMES]
    cA = np.bincount(P.frame.values[P.team.values == AWAY], minlength=N_VIDEO)[EVAL_FRAMES]
    m["overflow"] = (cH > 11) | (cA > 11)
    m["low_cov"] = m.cnt < 20
    m["minute"] = m.frame // (60 * FPS)
    return m.groupby("minute").agg(first_frame=("frame", "min"), overflow=("overflow", "mean"),
                                   low_coverage=("low_cov", "mean"), people=("cnt", "mean"))

hb = H2H[BEST].assign(minute=lambda x: x.frame // (60 * FPS))
mins = per_minute(ATD_T).add_prefix("atd_").join(per_minute(VARIANTS[BEST]).add_prefix("ours_"))
g = hb.groupby("minute")
mins["h2h_match_rate"] = g.matched.sum() / g.n_atd.sum().clip(lower=1)
mins["h2h_team_agreement"] = g.team_same.sum() / g.team_known.sum().clip(lower=1)
mins["atd_on_referee"] = g.atd_on_referee.mean()
mins["disagreement"] = 1 - mins.h2h_match_rate * mins.h2h_team_agreement
mins.round(3).to_csv(os.path.join(EVAL_DIR, "fault_map_per_minute.csv"))
print("Minutes with the most ATD-vs-ours disagreement:")
print(mins.sort_values("disagreement", ascending=False).head(10).round(2).to_string())


## Plots

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
C_ATD, C_OURS, C_BEST = "#8a8984", "#86b6ef", "#2a78d6"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED,
                     "ytick.color": INK, "font.size": 9, "axes.titlesize": 10, "axes.titlelocation": "left"})

metrics = [("coverage_20_23", "Coverage: frames with 20–23 people", "higher is better", "{:.0%}"),
           ("team_overflow", "Team overflow: frames with >11 on a team", "lower is better", "{:.1%}"),
           ("jumps_per_1000", "Jumps >12 m/s per 1,000 steps", "lower is better", "{:.1f}"),
           ("segments_per_half", "Track pieces per half", "lower is better", "{:,.0f}"),
           ("gk_sane", "Frames with one GK per team, own half", "higher is better", "{:.0%}"),
           ("h2h_team_agreement", "Team agreement with ATD", "higher = closer to ATD", "{:.0%}")]
order = ["ATD"] + list(SCORE.index[SCORE.index.str.endswith("+post")])
fig, axes = plt.subplots(2, 3, figsize=(15, 0.32 * len(order) * 2 + 2.5), sharey=True)
for ax, (col, title, better, fmt) in zip(axes.flat, metrics):
    vals = SCORE.loc[order, col]
    colors = [C_ATD if s == "ATD" else C_BEST if s == BEST else C_OURS for s in order]
    y = np.arange(len(order))
    ax.barh(y, vals.fillna(0), color=colors, height=0.7)
    for yi, v in zip(y, vals):
        ax.text(0, yi, "  n/a" if pd.isna(v) else "  " + fmt.format(v), va="center", ha="left",
                fontsize=8, color=INK)
    ax.set_yticks(y, order); ax.invert_yaxis()
    ax.set_title(f"{title}\n", fontsize=10); ax.text(0, 1.01, better, transform=ax.transAxes, color=MUTED, fontsize=8)
    ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
fig.suptitle(f"ATD (grey) vs our variants (blue); best = {BEST} (dark blue). Mode: {RUN_MODE}",
             x=0.01, ha="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(os.path.join(EVAL_DIR, "scorecard.png"), dpi=120, bbox_inches="tight"); plt.show()

# Fault strip: one row per signal, one column per minute (darker = worse)
strip = mins[["atd_overflow", "ours_overflow", "atd_low_coverage", "ours_low_coverage", "disagreement"]].T.values
fig, ax = plt.subplots(figsize=(15, 2.4))
im = ax.imshow(strip, aspect="auto", cmap=matplotlib.colors.LinearSegmentedColormap.from_list(
    "blue", ["#f4f8fd", "#86b6ef", "#2a78d6", "#0d366b"]), vmin=0, vmax=1, interpolation="nearest")
ax.set_yticks(range(5), ["ATD overflow", "Ours overflow", "ATD low coverage", "Ours low coverage",
                         "ATD vs ours disagreement"])
ticks = np.arange(0, len(mins), max(1, len(mins) // 20))
ax.set_xticks(ticks, mins.index[ticks]); ax.set_xlabel("video minute (frame // 1500)")
ax.set_title("Fault map per minute — share of the minute's frames affected", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.015, pad=0.01, format=matplotlib.ticker.PercentFormatter(1.0))
fig.savefig(os.path.join(EVAL_DIR, "fault_map.png"), dpi=120, bbox_inches="tight"); plt.show()


## Side-by-side review clips (ATD left, best variant right) from the most-disputed minutes

In [ ]:
BGR = {HOME: (214, 120, 42), AWAY: (52, 104, 235), NO_TEAM: (200, 200, 200)}   # blue / orange / grey
REF_BGR, GK_RING = (30, 30, 30), (0, 200, 255)


def draw_atd(fr, f):
    axy = ATD[f]
    ok = np.flatnonzero(np.isfinite(axy[:, 0]))
    u, v = pitch_to_pix(f, axy[ok, 0].astype(np.float64), axy[ok, 1].astype(np.float64))
    for s, x, y in zip(ok, u, v):
        if np.isfinite(x):
            cv2.circle(fr, (int(x), int(y)), 9, BGR[int(SLOT_TEAM[s])], -1)
            cv2.putText(fr, str(s + 1), (int(x) + 8, int(y) - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return fr


def draw_ours(fr, rows):
    for r in rows.itertuples():
        col = REF_BGR if r.role == ROLE_REF else BGR[int(r.team)]
        cx, w = int((r.x1 + r.x2) / 2), max(int((r.x2 - r.x1) / 2), 6)
        cv2.ellipse(fr, (cx, int(r.y2)), (w, max(w // 3, 3)), 0, -45, 235, col, 3)
        if r.role == ROLE_GK:
            cv2.ellipse(fr, (cx, int(r.y2)), (w + 4, max(w // 3, 3) + 3), 0, -45, 235, GK_RING, 2)
        cv2.putText(fr, f"{int(r.id) % 1_000_000}", (int(r.x1), int(r.y1) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
    return fr


if MAKE_CLIPS:
    T = VARIANTS[BEST]
    tb = np.searchsorted(T.frame.values, np.arange(N_VIDEO + 1))
    picked = []
    for m in mins.sort_values("disagreement", ascending=False).index:
        if all(abs(m - p) > 1 for p in picked):
            picked.append(m)
        if len(picked) == N_CLIPS:
            break
    n_clip = CLIP_SECONDS * FPS
    for m in picked:
        f0 = int(mins.loc[m, "ours_first_frame"])
        f1 = min(f0 + n_clip, int(EVAL_FRAMES[EVAL_FRAMES >= f0].max()) + 1)
        path = os.path.join(CLIP_DIR, f"minute_{m:03d}_frames_{f0}_{f1 - 1}.mp4")
        cap = cv2.VideoCapture(VIDEO); cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
        out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (VID_W, VID_H // 2))
        for f in range(f0, f1):
            ok, fr = cap.read()
            if not ok:
                break
            a = draw_atd(fr.copy(), f); b = draw_ours(fr.copy(), T.iloc[tb[f]:tb[f + 1]])
            cv2.putText(a, f"ATD  frame {f}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3)
            cv2.putText(b, f"{BEST}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3)
            out.write(np.hstack([cv2.resize(a, (VID_W // 2, VID_H // 2)), cv2.resize(b, (VID_W // 2, VID_H // 2))]))
        out.release(); cap.release()
        print("clip:", path)


## Export

In [ ]:
F = VARIANTS[BEST].copy()
F["track_id"] = F.id.astype(np.int64)
F["role"] = F.role.map(ROLE_NAME)
F["team"] = F.team.map(TEAM_LETTER)
F["team_p_home"] = F.p
cols = ["frame", "half", "track_id", "role", "team", "conf", "x1", "y1", "x2", "y2", "x_m", "y_m", "team_p_home"]
F[cols].to_parquet(os.path.join(WORK, f"final_tracks_{RUN_MODE}.parquet"), index=False)

summary = {"run_mode": RUN_MODE, "best_variant": BEST, "beats_atd": bool(SCORE.loc[BEST, "beats_atd"]),
           "scored_frames": int(EVAL.sum()), "detections": int(len(D)), "model": os.path.basename(MODEL_PATH),
           "best": SCORE.loc[BEST].drop(["beats_atd"]).astype(float).round(4).to_dict(),
           "atd": SCORE.loc["ATD"].drop(["beats_atd"]).astype(float).round(4).to_dict()}
save_json(summary, os.path.join(EVAL_DIR, f"summary_{RUN_MODE}.json"))
print(json.dumps(summary, indent=1))
print("\nOutputs in", WORK)
for p in sorted(glob.glob(os.path.join(WORK, "**", "*"), recursive=True)):
    if os.path.isfile(p) and not p.endswith(".parquet") or p.endswith(f"final_tracks_{RUN_MODE}.parquet"):
        print(f"  {os.path.relpath(p, WORK):60s} {os.path.getsize(p) / 1e6:8.1f} MB")
